# iSTAMP — example inference

Load the trained **GATv2** model and run subtype prediction on an example graph.

This notebook runs on a fresh clone using the bundled **synthetic** example graph
(`examples/data/synthetic_example_pyg.pt`, random features — predictions are meaningless, it only
demonstrates the pipeline). To predict on **real** cohort graphs, download the `Img_*_pyg.pt` files
(patient-derived; see the manuscript's Data Availability statement) and set the `RAMAN_DATA` environment variable.

In [ ]:
import os, sys, glob, numpy as np, torch
REPO = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO, "src", "istamp_analysis"))
from models import RamanGATv2MultiBias

CLASSES = ["GBM_MES_TYP", "GBM_MES_ATYP", "GBM_RTK1", "GBM_RTK2", "GBM_PNC"]
BIAS = {"Center": 4, "sample_type": 2, "image_id": 1274, "patient_id": 233}
MODEL_PATH = os.environ.get("RAMAN_MODEL", os.path.join(REPO, "models", "best_model_full.pt"))
print("model:", MODEL_PATH, "| exists:", os.path.exists(MODEL_PATH))

## 1. Load the trained model

In [ ]:
ck = torch.load(MODEL_PATH, weights_only=False, map_location="cpu"); p = ck["params"]
model = RamanGATv2MultiBias(
    in_channels=512, hidden_channels=p["hidden_multiplier"] * p["num_heads"], out_channels=5,
    edge_dim=1, num_heads=p["num_heads"], num_layers=p["num_layers"], dropout=p["dropout"],
    use_batch_norm=p["use_batch_norm"], bias_classes=BIAS,
    bias_hidden_dim=p["bias_hidden_dim"], grl_alpha=p["grl_alpha"])
model.load_state_dict(ck["model_state_dict"], strict=False); model.eval()
print(f"GATv2: heads={p['num_heads']} layers={p['num_layers']} "
      f"hidden={p['hidden_multiplier']*p['num_heads']} | frozen CONCH-512 node features")

## 2. Load an example graph
Prefer a real cohort graph if `RAMAN_DATA` points to the `Img_*_pyg.pt` files; otherwise fall
back to the bundled synthetic example.

In [ ]:
def softmax(z): z = z - z.max(); e = np.exp(z); return e / e.sum()
DATA_DIR = os.environ.get("RAMAN_DATA", "")
real = sorted(glob.glob(os.path.join(DATA_DIR, "Img_*_pyg.pt"))) if DATA_DIR else []
path = real[0] if real else os.path.join(REPO, "examples", "data", "synthetic_example_pyg.pt")
d = torch.load(path, weights_only=False, map_location="cpu")
print("loaded:", os.path.basename(path), "| image_id:", d.get("image_id"),
      "|", "SYNTHETIC" if "synthetic" in path else "REAL cohort graph")
graphs = d["pyg_dataset_by_hop"][3]        # hop-3 ego-subgraphs = the deployed configuration
print(f"{len(graphs)} hop-3 subgraphs, median nodes = {int(np.median([g.x.shape[0] for g in graphs]))}")

## 3. Predict — subgraph scores -> patient level
Patient/image prediction = **mean of the raw subgraph scores, then softmax** (the deployed aggregation).

In [ ]:
import pandas as pd
scores = []
with torch.no_grad():
    for g in graphs:
        g = g.clone(); g.batch = torch.zeros(g.x.shape[0], dtype=torch.long)
        scores.append(model(g)["pred"][0].numpy())
scores = np.stack(scores); agg = scores.mean(0); prob = softmax(agg)
print(pd.DataFrame({"class": CLASSES, "mean_score": agg.round(3), "prob": prob.round(3)})
      .sort_values("prob", ascending=False).to_string(index=False))
print("\nPredicted subtype:", CLASSES[int(agg.argmax())], "| confidence:", round(float(prob.max()), 3))

## 4. Interpretability — integrated-gradient node attribution
Which graph nodes drive the predicted subtype (zero baseline, 32 steps).

In [ ]:
g = graphs[0].clone(); g.batch = torch.zeros(g.x.shape[0], dtype=torch.long)
t = int(model(g)["pred"][0].argmax())
x = g.x.float(); base = torch.zeros_like(x); tot = torch.zeros_like(x)
for a in torch.linspace(0, 1, 32):
    xi = (base + a * (x - base)).detach().requires_grad_(True)
    gg = g.clone(); gg.x = xi
    s = model(gg)["pred"][0, t]
    tot += torch.autograd.grad(s, xi)[0].detach()
ig = ((x - base) * tot / 32).sum(1).numpy(); order = np.argsort(-np.abs(ig))
print("Target class:", CLASSES[t])
print("Top-5 most influential nodes (index : IG):",
      [(int(i), round(float(ig[i]), 3)) for i in order[:5]])

---
**Next:** run the interactive app with `streamlit run streamlit_app/app.py` (set `RAMAN_MODEL` to
`models/best_model_full.pt`). For the full analysis see `notebooks/01_master_inference.ipynb`.